# 03 – LSTM Model Training & Evaluation

This notebook covers:
1. Defining the `OrbitalLSTM` architecture
2. Training with Adam optimiser and early stopping
3. Random Forest baseline
4. Computing RMSE and MAE on the test set

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import (
    load_tle_file, build_dataset, create_windows, split_and_normalize,
    WINDOW_SIZE, STEP_MIN, HOURS
)
from src.model import (
    OrbitalLSTM, train_lstm, predict_lstm,
    RandomForestPredictor, compute_metrics
)

## 3.1 Prepare dataset

In [ ]:
tle_list = load_tle_file('../data/starlink_tle.txt')
records  = build_dataset(tle_list, hours=HOURS, step_min=STEP_MIN)
X, y     = create_windows(records, window_size=WINDOW_SIZE)
X_train, y_train, X_test, y_test, x_sc, y_sc = split_and_normalize(X, y)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 3.2 LSTM architecture

```
Input  →  [Batch, 12, 8]
   ↓
LSTM Layer 1  (64 hidden units)
   ↓
LSTM Layer 2  (64 hidden units)
   ↓
Linear(64 → 3)
   ↓
Output  →  [Batch, 3]
```

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = OrbitalLSTM(input_dim=8, hidden_dim=64, num_layers=2, output_dim=3)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total trainable parameters: {n_params:,}')

## 3.3 Train LSTM

In [ ]:
history = train_lstm(
    model, X_train, y_train,
    epochs=50, lr=1e-3, batch_size=64, patience=5,
    device=device
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train MSE')
plt.plot(history['val_loss'],   label='Val MSE')
plt.xlabel('Epoch')
plt.ylabel('MSE (normalised)')
plt.title('LSTM Training Curve')
plt.legend()
plt.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 3.4 LSTM evaluation

In [ ]:
y_pred_n  = predict_lstm(model, X_test, device=device)
y_pred_km = y_sc.inverse_transform(y_pred_n)
y_true_km = y_sc.inverse_transform(y_test)

lstm_m = compute_metrics(y_true_km, y_pred_km)
print(f'LSTM  RMSE : {lstm_m["RMSE_km"]:.3f} km')
print(f'LSTM  MAE  : {lstm_m["MAE_km"]:.3f} km')

## 3.5 Random Forest baseline

In [ ]:
rf = RandomForestPredictor(n_estimators=100)
rf.fit(X_train, y_train)
y_rf_n   = rf.predict(X_test)
y_rf_km  = y_sc.inverse_transform(y_rf_n)
rf_m     = compute_metrics(y_true_km, y_rf_km)
print(f'RF    RMSE : {rf_m["RMSE_km"]:.3f} km')
print(f'RF    MAE  : {rf_m["MAE_km"]:.3f} km')

## 3.6 Save model weights

In [ ]:
import torch
torch.save(model.state_dict(), '../lstm.pt')
print('Model saved → lstm.pt')